# Trekomend — CSV to SQLite Converter

Converts `TMDB_movie_dataset_v11.csv` (1.43M rows, ~632 MB) into a
**fast, indexed SQLite database** with FTS5 full-text search.

**Why SQLite?**
- Single file, zero setup on VPS
- Instant lookup by TMDB ID (for FAISS result mapping)
- Full-text search across titles, overviews, keywords, genres
- Filtered browsing (genre + year + rating + votes)

---

## Runtime (Colab Free Tier)

| Step | Time |
|---|---|
| Download CSV | ~2 min (632 MB) |
| Bulk insert | ~3-5 min |
| Create indexes | ~30 sec |
| Create FTS5 | ~2-3 min |
| **Total** | **~7-11 min** |

## Output

| File | Size | |
|---|---|---|
| `tmdb_movies.db` | ~300-400 MB | Movies table + indexes + FTS5 |

---

In [ ]:
# =====================================================================
# Cell 1 — Configuration
# =====================================================================
import csv
import os
import sqlite3
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────
CSV_URL = (
    "https://huggingface.co/datasets/fukitweball/TMDB/resolve/main/"
    "TMDB_movie_dataset_v11.csv"
)
CSV_PATH = Path("TMDB_movie_dataset_v11.csv")
DB_PATH = Path("tmdb_movies.db")

# ── Performance ────────────────────────────────────────────────────
CHUNK_SIZE = 100_000  # CSV rows per pandas chunk
BATCH_SIZE = 10_000   # Rows per SQLite INSERT batch

# ── Columns ────────────────────────────────────────────────────────
# All columns we want in the database (21 total)
ALL_COLUMNS = [
    "id", "title", "original_title", "release_date",
    "runtime", "original_language", "overview", "tagline",
    "genres", "keywords",
    "production_companies", "production_countries",
    "vote_average", "vote_count", "popularity",
    "budget", "revenue", "status", "adult",
]
# Derived columns (computed, not from CSV)
DERIVED_COLUMNS = ["year", "primary_genre"]

print(f"CSV source: {CSV_URL}")
print(f"Output DB:  {DB_PATH}")
print(f"Columns:    {len(ALL_COLUMNS)} direct + {len(DERIVED_COLUMNS)} derived = 21 total")
print(f"Chunk:      {CHUNK_SIZE:,} read / {BATCH_SIZE:,} insert")

In [ ]:
# =====================================================================
# Cell 2 — Download CSV (if not present)
# =====================================================================
if not CSV_PATH.exists():
    print(f"Downloading TMDB dataset (~632 MB)...")
    print(f"URL: {CSV_URL}")
    !wget -q -O "{CSV_PATH}" "{CSV_URL}"
    print(f"Downloaded: {CSV_PATH.stat().st_size / 1e6:.0f} MB")
else:
    print(f"CSV already exists: {CSV_PATH.stat().st_size / 1e6:.0f} MB")

# ── Count rows (fast) ─────────────────────────────────────────────
print("Counting rows...")
with open(CSV_PATH, "r", encoding="utf-8") as f:
    total_rows = sum(1 for _ in f) - 1  # minus header
print(f"Total rows in CSV: {total_rows:,}")

# ── Verify columns ─────────────────────────────────────────────────
csv_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
missing = [c for c in ALL_COLUMNS if c not in csv_cols]
if missing:
    print(f"WARNING: Columns missing from CSV: {missing}")
    ALL_COLUMNS = [c for c in ALL_COLUMNS if c in csv_cols]
    print(f"Using {len(ALL_COLUMNS)} available columns.")
else:
    print(f"All {len(ALL_COLUMNS)} expected columns present.")

In [ ]:
# =====================================================================
# Cell 3 — Helper Functions
# =====================================================================

def safe_value(val):
    """Convert to string or None for SQLite."""
    if val is None:
        return None
    if isinstance(val, float):
        if pd.isna(val) or not np.isfinite(val):
            return None
        return float(val)  # keep as float for numeric columns
    s = str(val).strip()
    if s.lower() in ("nan", "none", "null", ""):
        return None
    return s


def extract_year(date_val):
    """Extract year from release_date."""
    if date_val is None:
        return None
    if isinstance(date_val, float) and pd.isna(date_val):
        return None
    s = str(date_val).strip()
    if not s or s.lower() in ("nan", "none", "null", ""):
        return None
    if len(s) >= 4 and s[:4].isdigit():
        try:
            return int(s[:4])
        except ValueError:
            return None
    return None


def extract_primary_genre(genres_val):
    """Extract the first genre from comma-separated string."""
    if genres_val is None:
        return "Unknown"
    if isinstance(genres_val, float) and pd.isna(genres_val):
        return "Unknown"
    s = str(genres_val).strip()
    if not s or s.lower() in ("nan", "none", "null", ""):
        return "Unknown"
    parts = s.split(",")
    first = parts[0].strip()
    return first if first else "Unknown"


print("Helper functions ready.")

In [ ]:
# =====================================================================
# Cell 4 — Create Database & Import Data
# =====================================================================

if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Removed existing {DB_PATH}")

# ── Open with extreme performance pragmas ────────────────────────
db = sqlite3.connect(str(DB_PATH))
db.execute("PRAGMA journal_mode = OFF")      # No rollback journal
db.execute("PRAGMA synchronous = 0")          # No fsync
db.execute("PRAGMA cache_size = -300000")     # 300 MB cache
db.execute("PRAGMA temp_store = MEMORY")      # Temp tables in RAM
db.execute("PRAGMA locking_mode = EXCLUSIVE") # Single connection
db.execute("PRAGMA page_size = 65536")        # 64KB pages (fewer writes)

print("Performance pragmas set.")

# ── Create table ─────────────────────────────────────────────────
CREATE_SQL = """
CREATE TABLE movies (
    id                  INTEGER PRIMARY KEY,
    title               TEXT,
    original_title      TEXT,
    release_date        TEXT,
    year                INTEGER,
    runtime             REAL,
    original_language   TEXT,
    overview            TEXT,
    tagline             TEXT,
    genres              TEXT,
    primary_genre       TEXT,
    keywords            TEXT,
    production_companies TEXT,
    production_countries TEXT,
    vote_average        REAL,
    vote_count          REAL,
    popularity          REAL,
    budget              REAL,
    revenue             REAL,
    status              TEXT,
    adult               TEXT
)
"""
db.execute(CREATE_SQL)
print("Table 'movies' created.")

# ── Prepare INSERT statement ────────────────────────────────────
INSERT_SQL = """INSERT OR REPLACE INTO movies (
    id, title, original_title, release_date, year, runtime,
    original_language, overview, tagline, genres, primary_genre,
    keywords, production_companies, production_countries,
    vote_average, vote_count, popularity, budget, revenue, status, adult
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)"""

# ── Build a lookup: CSV column index -> transform function ───────
# We read CSV columns in order, apply transforms to build a row tuple
csv_header = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()

# Map each of our target columns to: (csv_index_or_None, transform_fn)
COLUMN_CONFIG = {
    "id":                     ("id", lambda x: int(float(x)) if not pd.isna(x) else None),
    "title":                  ("title", safe_value),
    "original_title":         ("original_title", safe_value),
    "release_date":           ("release_date", safe_value),
    "year":                   ("release_date", extract_year),  # derived
    "runtime":                ("runtime", safe_value),
    "original_language":      ("original_language", safe_value),
    "overview":               ("overview", safe_value),
    "tagline":                ("tagline", safe_value),
    "genres":                 ("genres", safe_value),
    "primary_genre":          ("genres", extract_primary_genre),  # derived
    "keywords":               ("keywords", safe_value),
    "production_companies":   ("production_companies", safe_value),
    "production_countries":   ("production_countries", safe_value),
    "vote_average":           ("vote_average", safe_value),
    "vote_count":             ("vote_count", safe_value),
    "popularity":             ("popularity", safe_value),
    "budget":                 ("budget", safe_value),
    "revenue":                ("revenue", safe_value),
    "status":                 ("status", safe_value),
    "adult":                  ("adult", safe_value),
}

# All 21 columns in INSERT order
INSERT_COLUMNS = [
    "id", "title", "original_title", "release_date", "year", "runtime",
    "original_language", "overview", "tagline", "genres", "primary_genre",
    "keywords", "production_companies", "production_countries",
    "vote_average", "vote_count", "popularity", "budget", "revenue",
    "status", "adult",
]

print("INSERT statement prepared.")

In [ ]:
# =====================================================================
# Cell 5 — Bulk Import (streaming from CSV)
# =====================================================================

print(f"Starting bulk import of {total_rows:,} rows...")
print(f"  Chunk size: {CHUNK_SIZE:,} | Batch size: {BATCH_SIZE:,}")
print()

total_inserted = 0
start_time = time.perf_counter()

reader = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False)

for chunk_num, chunk in enumerate(reader):
    n_chunk = len(chunk)
    
    # Build row tuples for this chunk
    rows = []
    for _, row in chunk.iterrows():
        row_tuple = tuple(
            COLUMN_CONFIG[col][1](row.get(COLUMN_CONFIG[col][0]))
            for col in INSERT_COLUMNS
        )
        rows.append(row_tuple)
    
    # Insert in batches
    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i:i + BATCH_SIZE]
        db.executemany(INSERT_SQL, batch)
    
    db.commit()
    total_inserted += n_chunk
    
    elapsed = time.perf_counter() - start_time
    rps = total_inserted / elapsed if elapsed > 0 else 0
    pct = 100 * total_inserted / total_rows
    eta = (total_rows - total_inserted) / rps / 60 if rps > 0 else 0
    
    print(f"  Chunk {chunk_num + 1}: {total_inserted:,}/{total_rows:,} "
          f"({pct:.1f}%) | {rps:,.0f} rows/s | ETA: {eta:.1f} min")

elapsed_total = time.perf_counter() - start_time
print(f"\nImport complete: {total_inserted:,} rows in {elapsed_total:.1f}s "
      f"({total_inserted/elapsed_total:,.0f} rows/s)")

# ── Verify count ─────────────────────────────────────────────────
count = db.execute("SELECT COUNT(*) FROM movies").fetchone()[0]
print(f"Verified: {count:,} rows in movies table")

In [ ]:
# =====================================================================
# Cell 6 — Create Indexes (after import = much faster)
# =====================================================================

print("Creating indexes...")
t_idx = time.perf_counter()

INDEXES = [
    ("idx_title",      "movies(title)"),
    ("idx_genre",      "movies(primary_genre)"),
    ("idx_year",       "movies(year)"),
    ("idx_votes",      "movies(vote_count)"),
    ("idx_popularity", "movies(popularity)"),
    ("idx_rating",     "movies(vote_average)"),
]

for name, definition in INDEXES:
    db.execute(f"CREATE INDEX IF NOT EXISTS {name} ON {definition}")
    print(f"  {name}")

db.commit()
dt = time.perf_counter() - t_idx
print(f"Indexes created in {dt:.1f}s")

In [ ]:
# =====================================================================
# Cell 7 — Create FTS5 Full-Text Search Index
# =====================================================================
# FTS5 indexes: title, overview, keywords, tagline, genres, primary_genre, year
# Uses 'content=' sync mode — auto-reads from movies table
# =====================================================================

print("Creating FTS5 full-text search index...")
print("  (this reads every row to build the index — ~2-3 min)")
t_fts = time.perf_counter()

db.execute("""
    CREATE VIRTUAL TABLE movies_fts USING fts5(
        title, overview, keywords, tagline, genres, primary_genre, year,
        content='movies',
        content_rowid='id'
    )
""")

# Populate FTS — 'rebuild' reads from the content table
db.execute("INSERT OR REPLACE INTO movies_fts(movies_fts) VALUES('rebuild')")
db.commit()

dt = time.perf_counter() - t_fts
print(f"FTS5 index created in {dt:.1f}s")

# ── Verify FTS ──────────────────────────────────────────────────
fts_count = db.execute("SELECT COUNT(*) FROM movies_fts").fetchone()[0]
print(f"FTS5 rows: {fts_count:,}")

In [ ]:
# =====================================================================
# Cell 8 — Restore Safe Settings & Optimize
# =====================================================================

# Switch back to WAL (safe for multi-reader)
db.execute("PRAGMA journal_mode = WAL")
db.execute("PRAGMA synchronous = NORMAL")
db.execute("PRAGMA locking_mode = NORMAL")

# Run ANALYZE for query planner statistics
print("Running ANALYZE...")
db.execute("ANALYZE")
db.commit()

# Vacuum to reclaim space and defragment
print("Running VACUUM...")
db.execute("VACUUM")
db.commit()

print("Database optimized.")

In [ ]:
# =====================================================================
# Cell 9 — Validation
# =====================================================================

print("=" * 60)
print("VALIDATION")
print("=" * 60)

# Row count
count = db.execute("SELECT COUNT(*) FROM movies").fetchone()[0]
fts_count = db.execute("SELECT COUNT(*) FROM movies_fts").fetchone()[0]
print(f"Movies:  {count:,}")
print(f"FTS5:    {fts_count:,}")

# Sample
row = db.execute(
    "SELECT id, title, primary_genre, year, vote_average, vote_count "
    "FROM movies LIMIT 5"
).fetchall()
print(f"\nSample rows:")
for r in row:
    print(f"  #{r[0]:<8} {str(r[1])[:45]:45s} {r[2]:15s} {r[3]}   "
          f"rating={r[4]}  votes={r[5]}")

# Indexes
idx_list = db.execute(
    "SELECT name FROM sqlite_master WHERE type='index' ORDER BY name"
).fetchall()
print(f"\nIndexes ({len(idx_list)}): {', '.join(r[0] for r in idx_list)}")

# Genre distribution
top_genres = db.execute(
    "SELECT primary_genre, COUNT(*) as c FROM movies "
    "GROUP BY primary_genre ORDER BY c DESC LIMIT 10"
).fetchall()
print(f"\nTop genres:")
for g, c in top_genres:
    print(f"  {g:20s} {c:>8,}")

# Year range
min_year = db.execute(
    "SELECT MIN(year) FROM movies WHERE year IS NOT NULL"
).fetchone()[0]
max_year = db.execute(
    "SELECT MAX(year) FROM movies WHERE year IS NOT NULL"
).fetchone()[0]
print(f"\nYear range: {min_year} – {max_year}")

# FTS5 test
for query in ["sci-fi", "romantic comedy", "crime thriller"]:
    try:
        results = db.execute(
            "SELECT m.title, m.primary_genre, m.year "
            "FROM movies_fts fts JOIN movies m ON m.id = fts.rowid "
            "WHERE movies_fts MATCH ? ORDER BY rank LIMIT 3",
            (query,)
        ).fetchall()
        print(f"
FTS5 '{query}':")
        for title, genre, year in results:
            print(f"  {str(title)[:50]:50s} {str(genre):15s} {str(year)}")
    except Exception as e:
        print(f"
FTS5 '{query}': {e}")

In [ ]:
# =====================================================================
# Cell 10 — Close & Download
# =====================================================================

db.close()
print(f"Database saved: {DB_PATH}")
print(f"Size: {DB_PATH.stat().st_size / 1e6:.1f} MB")

# ── Colab download link ───────────────────────────────────────────
try:
    from google.colab import files
    files.download(str(DB_PATH))
except ImportError:
    print(f"\nNot in Colab. File is at: {DB_PATH.absolute()}")
    print("Download it manually or use:")
    print(f"  cp {DB_PATH} /content/drive/MyDrive/")

---

## Usage After Download

```python
import sqlite3

db = sqlite3.connect("tmdb_movies.db")
db.row_factory = sqlite3.Row

# Lookup by TMDB ID
row = db.execute("SELECT * FROM movies WHERE id = ?", (27205,)).fetchone()
print(row["title"])  # Inception

# Full-text search
results = db.execute(
    """SELECT title, primary_genre, year
       FROM movies_fts
       WHERE movies_fts MATCH ? ORDER BY rank LIMIT 10""",
    ("sci-fi thriller",)
).fetchall()

# Filtered browse
results = db.execute(
    """SELECT title, year, vote_average FROM movies
       WHERE primary_genre = ? AND year >= ? AND vote_count >= ?
       ORDER BY popularity DESC LIMIT 20""",
    ("Sci-Fi", 2010, 100)
).fetchall()

db.close()
```

### Performance (1.43M rows)

| Query | Latency |
|---|---|
| Lookup by ID (indexed) | <0.1 ms |
| Lookup by title (LIKE) | 0.5-2 ms |
| FTS5 search | 0.5-5 ms |
| Filtered browse | 1-5 ms |
| Genre list | <0.5 ms |

---